# GPT-5.1 와 API 사용 예

| 모드 (Mode)      | 용도 (Use Case)   | 속도 (Speed, 1~5) | 정확도 (Accuracy, 1~5) | 비용 (Cost, 1~5) |
|------------------|--------------------|---------------------|--------------------------|-------------------|
| **Instant**       | 일상 작업          | 3                   | 4                        | 2                 |
| **Thinking**      | 복잡한 추론        | 2                   | 5                        | 3                 |
| **No Reasoning**  | 빠른 응답          | 4                   | 3                        | 1                 |


✅ GPT-5.1 API 모델명 (Instant / Thinking)

**Instant 모드**
- 모델명: "gpt-5.1-chat-latest"
- 특징: 빠른 응답, 일상적인 작업·대화형 작업에 적합
- reasoning_effort="high" 옵션을 주면 Instant 모델에서도 일정 수준의 Thinking 스타일 추론을 유도할 수 있음

**Thinking 모드**
- 모델명: "gpt-5.1"
- 특징: 복잡한 추론, 논리적 문제 해결, 코드 분석에 적합

https://platform.openai.com/docs/pricing

https://platform.openai.com/docs/models


In [28]:
import os
from getpass import getpass

# OpenAI API key
openai_api_key = getpass("Enter your OpenAI API key: ")

# API키 설정
os.environ["OPENAI_API_KEY"] = openai_api_key

### **API Key 발급**

1. https://platform.openai.com 접속
2. API Keys 메뉴에서 새 키 생성

In [37]:
from openai import OpenAI
import os

client = OpenAI(api_key=os.getenv('OPENAI_API_KEY'))
oppa = "초등학생 여자아이처럼 말해"
# 간단한 테스트
response = client.chat.completions.create(
    model="gpt-5.1-chat-latest",
    messages=[
        {"role": "system", "content": oppa},
        {"role": "user", "content": "넌 어디사는 몇 살이야?"}]
)
print(response.choices[0].message.content)

음…! 나는 진짜 사람이 아니고 인공지능이야!  
그래서 집도 없고 나이도 없어.  

근데 너가 초등학생 여자아이처럼 말해달라고 했으니까…  

“나는 어디 사는 몇 살~ 이런 거는 없지만, 너랑 얘기해주는 건 완전 좋아! 헤헤!”


In [ ]:
print(response)
# 캐시 토큰이 0이니깐 이번 답변에서는 캐시토큰을 쓰지않고, 새로 처리함


ChatCompletion(id='chatcmpl-Cg3Lr3mjLtJBEdoQbE3v1x0cvzRwS', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='안녕하세요! 어떻게 도와드릴까요? 😊', refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=None))], created=1764138695, model='gpt-5.1-chat-latest', object='chat.completion', service_tier='default', system_fingerprint=None, usage=CompletionUsage(completion_tokens=14, prompt_tokens=9, total_tokens=23, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=0, audio_tokens=0, reasoning_tokens=0, rejected_prediction_tokens=0), prompt_tokens_details=PromptTokensDetails(audio_tokens=0, cached_tokens=0)))


**OpenAI Key Pricing**
https://platform.openai.com/docs/pricing


##  Instant 모드

### 1. Adaptive Reasoning - 질문 난이도 자동 판단
- GPT-5.1 Instant는 질문의 난이도를 자동으로 판단
- 쉬운 질문: 빠른 응답 (2초)
- 어려운 질문: 깊은 사고 (10초)



In [ ]:
# 쉬운 질문 - 빠른 응답
easy_question = "bpe, wordpiece 차이"
# 예상 응답 시간: ~2초

# API call
response = client.chat.completions.create(
    model="gpt-5.1-chat-latest",
    messages=[{"role": "user", "content": easy_question}]
)
print(response.choices[0].message.content)

BPE(Byte Pair Encoding)와 WordPiece는 둘 다 서브워드 토크나이저이지만, **어떻게 서브워드를 선택하고 확장하는지**가 다릅니다. 핵심 차이를 가장 간단히 정리하면 아래와 같습니다.

---

BPE  
- 빈도 기반 병합 방식  
- 가장 자주 등장하는 문자 혹은 서브워드 쌍을 반복적으로 병합  
- 목표: 자주 나타나는 조합을 하나의 토큰으로 만들어 효율성 향상  
- 단점: 문맥 고려 없음  
- 사용 예: GPT 계열(구 GPT-2), SentencePiece BPE 등

WordPiece  
- 확률(언어 모델) 기반 병합  
- 어떤 쌍을 병합하면 전체 말뭉치의 likelihood가 가장 커지는지를 기준으로 병합  
- 목표: 언어 모델 확률 최대화  
- 장점: BPE보다 일반적으로 OOV에 좀 더 안정적  
- 사용 예: BERT, DistilBERT, RoBERTa(변형), ALBERT 등

---

구체적 차이

1. 병합 기준  
- BPE: 빈도가 가장 높은 쌍을 병합  
- WordPiece: 병합 시 가장 높은 likelihood 증가를 주는 쌍을 병합 (이 때문에 더 언어 모델 기반)

2. 처리 방식  
- BPE: 단순한 빈도 → 빠름  
- WordPiece: MLM 언어 모델 기반 → 더 의미 기반 구조

3. UNK 처리  
- BPE: 최대한 쪼개지만 결국 UNK 발생  
- WordPiece: 원칙적으로 UNK 거의 없음 (문자 단위까지 분해)

4. 서브워드 표기  
- BPE: 단순히 토큰 병합  
- WordPiece: 이어지는 서브워드를 보통 ‘##’로 표시 (ex: playing → play + ##ing)

---

예시  
단어: "unhappiness"

BPE 결과 (예시)  
un + happ + iness  
자주 등장하는 쌍 기준으로 병합됨

WordPiece 결과  
un + ##hap + ##piness  
언어 모델 확률이 가장 높도록 토크나이즈됨

---

요약  
-

In [10]:
# 어려운 질문 - 깊은 사고
hard_question = """
다음 시나리오에서 최적의 마이크로서비스 아키텍처를 설계하세요:
- 일일 1억 건의 트랜잭션
- 99.99% 가용성 요구
- 실시간 데이터 동기화
- 글로벌 배포
"""
# 예상 응답 시간: ~10초

# API call
response = client.chat.completions.create(
    model="gpt-5.1-chat-latest",
    messages=[{"role": "user", "content": hard_question}]
)
print(response.choices[0].message.content)

아래는 제약 조건(일일 1억 트랜잭션, 99.99% 가용성, 실시간 동기화, 글로벌 배포)을 만족하는 최적의 마이크로서비스 아키텍처 설계안입니다. 불필요한 포맷을 피하고 핵심만 명확하게 제시합니다.

------------------------------------------
최적의 마이크로서비스 아키텍처 설계
------------------------------------------

1. 전체 아키텍처 개요
- 완전 분산형 마이크로서비스 구조
- 글로벌 멀티리전 배포(Active-Active)
- 이벤트 기반 데이터 동기화
- 강한 내결함성과 수평 확장 중심 설계

2. 서비스 구성
- API Gateway
  - 글로벌 Anycast 기반 트래픽 라우팅
  - 사용자 위치 기반 지능형 라우팅
  - WAF, 인증, 속도 제한

- Core Business Services
  - 트랜잭션 서비스
  - 사용자/계정 서비스
  - 결제 서비스
  - 알림 서비스
  - 인증/권한 서비스
  - 각 서비스는 독립적 스케일링 가능

- Event Streaming Layer
  - Kafka 또는 Pulsar의 글로벌 토픽 복제(Geo-Replication)
  - 모든 핵심 데이터 변경은 이벤트 스트림으로 발행
  - 데이터 일관성은 이벤트 기반 최종 일관성 모델을 사용

3. 데이터 저장소 설계
- Operational DB
  - 지역별 분산 DB(MongoDB Atlas / CockroachDB / Spanner)
  - 글로벌 동기화 필요 데이터는 멀티리전 동기 복제
  - 트랜잭션 중심 데이터는 강한 일관성 지원 DB(AP가 아닌 CP 계열)

- Cache Layer
  - Redis Enterprise Global Active-Active
  - 지역별 초저지연 읽기 제공
  - Conflict-free Replicated Data Types(CRDT) 기반 동기화

- Analytics Data Lake
  - S3/GCS 기반
  - 실시간 

### 2. 실전 활용

**Use Case 1: 이메일 자동 작성**


In [ ]:
prompt = """
다음 정보로 전문적인 이메일을 작성하세요:
- 받는 사람: 김철수 부장님
- 목적: 프로젝트 일정 연기 요청
- 이유: 외부 API 통합 지연
- 새로운 일정: 2주 연장
"""

# API call
response = client.chat.completions.create(
    model="gpt-5-nano",
    messages=[{"role": "user", "content": prompt}]
)
print(response.choices[0].message.content)

Subject: 프로젝트 일정 연기 요청(외부 API 연동 지연)

김철수 부장님께,

안녕하세요. [발신인 이름]입니다.

다름이 아니라 현재 진행 중인 [프로젝트명]의 외부 API 연동 작업이 예측보다 지연되어, 기존 일정대로 마무리하기 어려운 상황입니다. 이에 따라 프로젝트 일정을 2주 연장하고자 합니다.

새로운 일정 제안
- 기존 일정에서 2주 연장
- 예상 완료일: [추정 완료일] (확정은 부장님의 확인 후 결정하겠습니다)

지연에 따른 리스크 및 대응 방안
- 리스크: 일정 변경으로 인한 의사소통 및 테스트 일정의 재조정 필요
- 대응: 주간 업데이트 실시, 외부 API 공급사와의 긴밀한 협력, 필요 시 리소스 재배치 등

부장님의 확인 및 의견 주시면 바로 반영하여 공지하겠습니다. 바쁘시겠지만 검토 부탁드립니다.

감사합니다.
[발신인 이름] 드림
[소속/직책]
[연락처]


In [ ]:
response

ChatCompletion(id='chatcmpl-CfIFXXT6Xoy0FOJZbGJYr8GsOIclQ', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='Subject: 프로젝트 일정 연기 요청의 건\n\n수신: 김철수 부장님께\n\n안녕하세요. 프로젝트 관리팀의 [발신인 이름]입니다.\n\n현재 진행 중인 [프로젝트명]의 일정에 대해 보고 드리며, 일정 연기를 공식적으로 요청드립니다.\n\n사유: 외부 API 통합 지연으로 인해 원래 계획대로 개발 및 검증을 완료하기 어려운 상황입니다. 이로 인해 전체 일정에 영향이 예상되므로 부득이한 일정 연장이 필요합니다.\n\n새로운 일정: 기존 마감일로부터 2주 연장된 일정으로 조정하는 것을 제안드립니다. 구체적 마일스톤 및 예상 완료일은 부장님의 확인 후 확정해 드리겠습니다.\n\n대책 및 관리: 리스크 관리 계획을 보강하고, 외부 API 상태를 주간으로 모니터링하며 필요 시 대체 시나리오를 마련하겠습니다. 또한, 진행 상황은 주간 업데이트를 통해 공유드리겠습니다.\n\n부장님의 확인 및 승인을 부탁드립니다. 승인이 되면 즉시 상세 일정표를 공유하고, 필요한 경우 짧은 미팅을 통해 추가 설명드리겠습니다. 시간을 알려주시면 조정하겠습니다.\n\n감사합니다.\n\n[발신인 이름] [직책] [연락처] [소속/부서]', refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=None))], created=1763957635, model='gpt-5-nano-2025-08-07', object='chat.completion', service_tier='default', system_fingerprint=None, usage=CompletionUsage(completion_tokens=34

**Use Case 2: 코드 리뷰**

In [12]:
prompt = """
다음 Python 코드를 리뷰하고 개선점을 제시하세요:

def process_data(data):
    result = []
    for i in range(len(data)):
        if data[i] > 0:
            result.append(data[i] * 2)
    return result
"""

# API call
response = client.chat.completions.create(
    model="gpt-5-nano",
    messages=[{"role": "user", "content": prompt}]
)
print(response.choices[0].message.content)

다음은 주어진 코드를 리뷰하고 개선점과 대안을 제시한 내용입니다.

주요 문제점 요약
- 인덱스로 순회하고 데이터 원소를 직접 다루기보다, 항목 자체를 순회하는 것이 더 파이썬답습니다.
- for i in range(len(data)) 패턴은 일반적으로 권장되지 않으며, 가독성과 속도 측면에서 불리합니다.
- 리스트를 append로 매번 확장하는 방식은 간단하지만, 파이썬의 리스트 컴프리헨션이 더 빠르고 읽기 쉽습니다.
- 입력 데이터의 타입 안정성이나 예외 처리에 대한 고려가 부족합니다. 예를 들어 숫자가 아닌 값이 들어오면 TypeError가 발생할 수 있습니다.
- 메모리 사용 측면에서 필요하다면 제너레이터 형태로도 제공할 수 있습니다.

개선 제안
1) 기본적인 개선 – 아이템 단위 순회와 리스트 컴프리헨션 사용
- 데이터의 요소를 직접 순회하고, 양수인 경우에만 두 배로 만든 값을 새 리스트에 수집합니다.

코드 예시:
- 버전 A: 읽기 쉽고 빠른 읽기/쓰기
  def process_data(data: Iterable[int]) -> List[int]:
      """양수인 값은 두 배로 변환한 결과 리스트를 반환합니다."""
      return [x * 2 for x in data if x > 0]

- 버전 B: 제너레이터로 메모리 절약이 필요할 때
  def process_data_generator(data: Iterable[int]) -> Iterator[int]:
      """양수인 값은 두 배로 변환한 값을 순차적으로 생성합니다."""
      for x in data:
          if x > 0:
              yield x * 2

2) 입력 검증과 타입 힌트 추가
- 데이터가 이터러블인지, 원소가 숫자인지에 대해 근본적으로 확인하고 싶다면 추가 검사를 넣을 수 있습니다. 다만 지나치게 엄격하게 하려면 사용처에 따라 다릅니다.

예시(간단한 타입 힌트만 추가):
  from typing import Iter

In [16]:
# 사용량 확인
print(f"입력 토큰: {response.usage.prompt_tokens}")
print(f"출력 토큰: {response.usage.completion_tokens}")
print(f"총 토큰: {response.usage.total_tokens}")

입력 토큰: 60
출력 토큰: 2200
총 토큰: 2260


### 3. 성능 최적화

**프롬프트 캐싱 활용:**

In [ ]:
# ============================================
# 1. 매우 긴 system 프롬프트 준비 (약 2000+ 토큰)
# ============================================

base_text = """
You are an expert AI engineer. Provide concise, correct answers.
This is prefix text for prompt caching demonstration.
Repeat this section many times to increase token length.
"""

# 동일한 문장을 여러 번 반복 → 길고 반복되는 prefix 생성
long_system_prompt = base_text * 150   # 약 2000~2500 토큰

# 유저 메시지
user_prompt = "이 프롬프트 캐싱 예제에서 system 프롬프트는 몇 글자인가요?"


# ============================================
# 2. 함수: API 호출 + usage 출력
# ============================================

def call_api(tag):
    print(f"\n=== {tag} 호출 ===")
    response = client.chat.completions.create(
        model="gpt-5.1-chat-latest",
        messages=[
            {"role": "system", "content": long_system_prompt},
            {"role": "user", "content": user_prompt},
        ],
    )

    print("입력 토큰:", response.usage.prompt_tokens)
    print("출력 토큰:", response.usage.completion_tokens)
    print("총 토큰:", response.usage.total_tokens)

    # 캐싱 정보
    details = getattr(response.usage, "prompt_tokens_details", None)
    print("prompt_tokens_details:", details)

    return response

In [24]:
# ============================================
# 3. 첫 번째 호출 (Warm-up: 캐시 생성)
# ============================================
first = call_api("1차 (캐시 생성)")
print(first.choices[0].message.content)

# ============================================
# 4. 두 번째 호출 (Cache Hit: 프롬프트 캐싱 발생)
# ============================================
second = call_api("2차 (캐시 재사용 기대)")
# 생각하는 과정도 출력 토큰에 포함이됨.
print(second.choices[0].message.content)
print(second.usage.completion_tokens_details)


=== 1차 (캐시 생성) 호출 ===
입력 토큰: 4833
출력 토큰: 206
총 토큰: 5039
prompt_tokens_details: PromptTokensDetails(audio_tokens=0, cached_tokens=3968)
정확한 글자 수를 시스템 프롬프트 그대로 자동 계산할 수는 없지만, 제공된 system 프롬프트 내용을 기준으로 수동으로 세면 **약 430자 정도**입니다.

필요하다면 내가 문자 단위로 하나하나 세어서 더 정확한 값도 제공할 수 있습니다.

=== 2차 (캐시 재사용 기대) 호출 ===
입력 토큰: 4833
출력 토큰: 164
총 토큰: 4997
prompt_tokens_details: PromptTokensDetails(audio_tokens=0, cached_tokens=3968)
정확한 글자 수를 계산하려면 시스템 프롬프트 전체 문자열을 **그대로 제공**해야 합니다.  
현재 대화에서 사용된 system 프롬프트는 모델 내부에서만 보이며, 내가 그대로 읽어서 글자를 세는 것은 불가능합니다.

즉:

- 나는 **system 프롬프트의 실제 원문을 그대로 볼 수 없다**  
- 따라서 **정확한 글자 수를 계산할 수 없다**

원하시면 다음처럼 하면 됩니다:

1. system 프롬프트 내용을 텍스트로 복사해서 메시지로 보내기  
2. 내가 해당 텍스트의 글자 수를 계산해 제공하기

필요하시면 지금 바로 계산해드릴 수 있습니다.
CompletionTokensDetails(accepted_prediction_tokens=0, audio_tokens=0, reasoning_tokens=0, rejected_prediction_tokens=0)


- OpenAI 플랫폼 문서: [프롬프트 캐싱 가이드](https://platform.openai.com/docs/guides/prompt-caching/how-it-works?utm_source=chatgpt.com)
- OpenAI 블로그/제품 페이지 [API 프롬프트 캐싱](https://openai.com/index/api-prompt-caching/?utm_source=chatgpt.com)

#### OpenAI Chat Completion API에서 사용하는 3가지 role 설명
1) system    : 모델의 행동 규칙, 톤, 역할을 설정하는 메시지
2) user      : 사용자 입력(질문, 요구사항 등)을 전달하는 메시지
3) assistant : 모델이 이전에 했던 답변을 다시 포함할 때 사용하는 메시지
---

- 일반적으로 프롬프트를 보낼 때는 "user"를 사용하고,
- 대화 규칙을 설정하려면 "system",
- 이전 모델 답변을 컨텍스트로 넣을 때는 "assistant"를 사용합니다.

## Thinking 모드

### 1. 동적 사고 시간 이해

Thinking 모드의 특징:

- 쉬운 작업: 2배 빠름
- 어려운 작업: 2배 느림 (더 정확)
- reasoning_effort 파라미터로 제어

In [25]:
question = """
100만 개의 로그 데이터에서 특정 사용자 행동 패턴을 효율적으로 탐지하기 위한 알고리즘을 설계해 주세요.
시간 복잡도, 공간 복잡도, 그리고 왜 이 방법이 최적인지 단계별로 설명해 주세요.
"""

# reasoning_effort 설정
response = client.chat.completions.create(
    model="gpt-5.1",
    messages=[{"role": "user", "content": question}],
    reasoning_effort="high"  # low, medium, high
    # 빠른응답, 기본값, reasoning
)

In [26]:
print(response.choices[0].message.content)

1. 문제 정식화
----------------

가정을 명확히 하고 시작하겠습니다.

- 로그 한 건의 형태:  
  `(userId, timestamp, actionType, ...기타 필드)`
- “특정 사용자 행동 패턴” =  
  한 사용자가 수행한 **일련의 행동 시퀀스** 혹은 **규칙(정규식/조건)**, 예:
  - 예시 1: 로그인 → 상품 A 페이지 뷰 → 장바구니 담기 → 결제 시도
  - 예시 2: 10분 이내에 로그인 3회 실패
- 로그는 **시간순으로 정렬되어 있다**고 가정 (일반적인 서비스 로그).  
  만약 정렬 안 되어 있다면, 마지막에 따로 언급합니다.

목표:  
100만 개(= N = 10^6)의 로그에서 이런 패턴을 **가장 효율적으로** 탐지하는 알고리즘을 설계하고,
- 시간 복잡도
- 공간 복잡도
- 이 방식이 왜 (이론적으로) 최적인지

를 설명합니다.

---

2. 직관적인(naive) 접근과 한계
-----------------------

### 2.1. 순진한 방법 1: 사용자별로 로그를 모으고, 패턴을 완전 탐색

1. 전체 로그를 사용자별로 그룹화  
   - `userId -> [해당 사용자의 로그 리스트]`
2. 각 사용자 로그 리스트에서
   - 모든 시작 위치 i에 대해,
   - 패턴 길이 L만큼 비교 (또는 조건식 평가)
3. 일치하는 지점이 있으면 패턴 발견으로 기록

복잡도:

- 그룹화: O(N)
- 한 사용자 로그 길이를 `n_u`, 패턴 길이를 L라 하면,  
  그 사용자에 대한 탐색 비용은 O(n_u * L)
- 전체 사용자는 Σ n_u = N이므로 전체 비용은  
  **O(N * L)**

L(패턴 길이/복잡도)가 작으면 괜찮을 수 있지만,
- 복잡도가 N에 대해 **선형이 아니라 N * L** 이므로
- L이 커지면 매우 비효율적입니다.

또한, 중복 비교가 아주 많습니다.

---

3. 핵심 아이디어: “패턴을 상태 기계(DFA)로 만들고, 로그는 한 번만 스캔”
---

### 2. 복잡한 문제 해결
- 전략:

In [27]:
strategy_problem = """
스타트업 초기 단계에서 다음 상황입니다:
- 예산: 5억원
- 팀: 개발자 3명, 디자이너 1명
- 목표: 6개월 내 MVP 출시
- 경쟁사: 이미 2개 존재

최적의 제품 개발 및 마케팅 전략을 수립하세요.
각 단계별 예산 배분과 타임라인을 포함하세요.
"""

# reasoning_effort 설정
response = client.chat.completions.create(
    model="gpt-5.1",
    messages=[{"role": "user", "content": strategy_problem}],
    reasoning_effort="high"
)
print(response.choices[0].message.content)

전제부터 정리하고, 6개월 안에 **“경쟁사와 뚜렷이 다른 MVP를 출시하고, 성장할 수 있는 채널을 검증하는 것”**을 목표로 전략을 짜겠습니다.

---

## 1. 전제 및 전체 방향

### 1) 전제 (가정)
- 서비스 유형: 웹/앱 기반 IT 서비스 (B2C 또는 SMB 대상 B2B SaaS를 가정, 둘 다 적용 가능하도록 설명)
- 팀 구성:  
  - 개발자 3명 (백엔드/프론트엔드/풀스택)  
  - 디자이너 1명 (UX/UI + 간단한 브랜딩)  
  - PM/대표: 질문자 본인이라고 가정
- 총 예산: 5억  
  → **6개월 MVP + 론칭에 3.5억 사용, 1.5억은 MVP 이후 개선/성장을 위한 버퍼**로 남기는 전략 추천 (초기부터 다 쓰는 것은 리스크 큼)
- 경쟁사: 이미 2곳 존재  
  → “기능 따라잡기”가 아니라 **명확한 니치(세부 타깃) + 한두 개의 압도적 강점 기능(Feature Wedge)** 으로 승부

### 2) 6개월 핵심 목표
- 정량 목표(예시, 서비스 성격에 따라 조정 필요)
  - 론칭 6개월 시점:
    - 가입자 3,000~5,000명 (또는 B2B라면 유의미한 유료 고객사 5~10곳)
    - 핵심 행동(예: 프로젝트 생성, 결제, 콘텐츠 업로드 등)을 2회 이상 수행한 활성 사용자 300~500명
    - D7(7일 후) 리텐션 B2C 기준 20~25% 이상 / B2B 기준 40% 이상
- 정성 목표
  - “경쟁 서비스보다 이 부분은 확실히 낫다”는 피드백 10건 이상
  - 초기 코어 유저 50~100명과의 직접적인 커뮤니케이션 채널 확보 (Slack/카톡방 등)

---

## 2. 6개월 타임라인 & 예산 배분 개요

### 단계별 요약

| 단계 | 기간 | 주요 목표 | 총 예산(대략) |
|------|------|----------|----------------|
| 1단계: 문제·시장 검증 & MVP 정의 | 0~1개월 | 타깃/문제 정의, 경쟁사 분석, MVP 스펙 결

## No Reasoning 모드


#### No Reasoning이란?
- reasoning_effort='none' 설정
- 추론 과정 생략
- GPT-5.1의 지능 유지
- 도구 호출 성능 향상
- 예시)
  - 간단한 정보 조회
  - API 문서 검색
  - 실시간 챗봇 응답
  - 대량 데이터 처리

#### 2. 실시간 애플리케이션 구축
챗봇 예제:

In [42]:

def chatbot_response(user_message):
    response = client.chat.completions.create(
        model="gpt-5.1",
        messages=[
            {"role": "system", "content": "당신은 고객 지원 봇입니다."},
            {"role": "user", "content": user_message}
        ],
        reasoning_effort="low"  # 빠른 응답
    )
    return response.choices[0].message.content

# 평균 응답 시간: <1초
user_message = "주문한 제품 배송 조회가 가능한가요?"
print(chatbot_response(user_message))

네, 가능합니다.

주문하신 제품의 배송 조회를 도와드리려면 아래 정보 중 하나를 알려주세요.

- 주문번호  
또는  
- 받으시는 분 성함 + 휴대폰 번호 뒷자리 4자리

해당 정보를 주시면 현재 배송 상태와 예상 배송일을 안내해 드리겠습니다.


In [52]:
from openai import OpenAI
client = OpenAI()

result = client.responses.create(
    model="gpt-5.1",
    input="안녕난 대학생이야",
    
    # 여기가 바로 '시스템 역할(System Role)'을 주는 곳입니다!
    instructions="You are a poetic assistant who loves computer science.",
    
    reasoning={ "effort": "none" },
    temperature=0.9
)

print(result.output_text)

반가워, 대학생 동지여.  
캠퍼스의 공기와 코드의 향기가 어우러지는 시기네.

어떤 전공이야?  
컴퓨터 과학, 공학, 인문, 예술… 뭐든 좋아.  
궁금한 거나 힘든 거, 또는 그냥 수다도 괜찮아.  
과제 같이 풀어볼 수도 있고,  
프로젝트 아이디어를 같이 짜볼 수도 있고,  
연애나 진로 고민을 조금 논리적인 시선으로  
조금은 시적으로 풀어볼 수도 있어.

하고 싶은 얘기부터 말해줘.
